# StoreOps Copilot: QLoRA fine-tune of Qwen2.5-1.5B-Instruct (Kaggle T4)

**Before running:** Settings → Accelerator **GPU T4 x2**, Internet **on**. Add-ons → Secrets: `HF_TOKEN`
(a Hugging Face **write** token) and optionally `WANDB_API_KEY`. Add the dataset containing
`train_text.jsonl` and `val_text.jsonl` (made locally by `python -m train.render`).

1. Run with `MAX_STEPS = 50` first (sanity check: the model must emit well-formed `<tool_call>` blocks).
2. Then set `MAX_STEPS = -1` (one full epoch, ~460 steps) and run all again.

In [ ]:
!pip install -q unsloth

In [ ]:
# ---- settings -------------------------------------------------------------------------------
MAX_STEPS = 50            # 50 = sanity check; -1 = full run (NUM_EPOCHS)
NUM_EPOCHS = 1
HF_REPO = "sidpnyc/storeops-copilot-qwen2.5-1.5b"   # GGUF repo; the LoRA adapter goes to HF_REPO + "-lora"
BASE = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"
MAX_SEQ = 4096
SEED = 42

In [ ]:
import glob, json, os, random
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")
try:
    os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
    REPORT_TO = "wandb"
    os.environ["WANDB_PROJECT"] = "storeops-copilot"
except Exception:
    REPORT_TO = "none"
print("logging to", REPORT_TO)

train_path = glob.glob("/kaggle/input/**/train_text.jsonl", recursive=True)[0]
val_path = glob.glob("/kaggle/input/**/val_text.jsonl", recursive=True)[0]
print(train_path, val_path)

In [ ]:
import torch
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE, max_seq_length=MAX_SEQ, dtype=None, load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=SEED,
)

In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files={"train": train_path, "validation": val_path})
print(ds)
# The chat template is already applied (locally, with Qwen's own template). Check one example.
print(ds["train"][0]["text"][-1200:])

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

args = SFTConfig(
    dataset_text_field="text",
    max_seq_length=MAX_SEQ,
    packing=False,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=2,
    num_train_epochs=NUM_EPOCHS,
    max_steps=MAX_STEPS,
    learning_rate=2e-4,
    lr_scheduler_type="linear",
    warmup_ratio=0.05,
    optim="adamw_8bit",
    weight_decay=0.0,
    fp16=not torch.cuda.is_bf16_supported(),   # T4: fp16
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50 if MAX_STEPS == 50 else 100,
    save_strategy="no",
    seed=SEED,
    output_dir="outputs",
    report_to=REPORT_TO,
)
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds["train"],
                     eval_dataset=ds["validation"], args=args)
# Loss only on assistant turns (tool calls + final answers). Qwen renders tool results as user
# turns, so they are masked too.
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n",
                                  response_part="<|im_start|>assistant\n")

# Masking check: decoded labels of one example should contain only assistant text.
labels = trainer.train_dataset[0]["labels"]
kept = [t for t in labels if t != -100]
print("trained tokens:", len(kept), "of", len(labels))
print(tokenizer.decode(kept)[:600])

In [ ]:
stats = trainer.train()
print(stats)

In [ ]:
# Loss curve for the README
import matplotlib.pyplot as plt
hist = trainer.state.log_history
json.dump(hist, open("log_history.json", "w"), indent=1)
tr = [(h["step"], h["loss"]) for h in hist if "loss" in h]
ev = [(h["step"], h["eval_loss"]) for h in hist if "eval_loss" in h]
plt.figure(figsize=(6, 3.5))
plt.plot(*zip(*tr), label="train")
if ev: plt.plot(*zip(*ev), "o-", label="validation")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.title("Qwen2.5-1.5B QLoRA"); plt.tight_layout()
plt.savefig("loss_curve.png", dpi=150); plt.show()

In [ ]:
# Sanity check: 5 held-out prompts (one per request kind). Tool calls must be well-formed JSON.
import re
FastLanguageModel.for_inference(model)
val = [json.loads(l) for l in open(val_path)]
picked = {}
for r in val:
    if r["source"] == "store" and r["kind"] not in picked:
        picked[r["kind"]] = r
ok = 0
for kind, r in picked.items():
    prompt = r["text"].split("<|im_start|>assistant\n")[0] + "<|im_start|>assistant\n"
    ids = tokenizer(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**ids, max_new_tokens=200, do_sample=False)
    gen = tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=False)
    calls = re.findall(r"<tool_call>\s*(.*?)\s*</tool_call>", gen, re.S)
    valid = all(isinstance(json.loads(c), dict) and "name" in json.loads(c) for c in calls) if calls else None
    good = valid if kind in ("lookup", "planning", "action") else not calls
    ok += bool(good)
    print(f"--- {kind}: {'OK' if good else 'CHECK'}\n{gen[:400]}\n")
print(f"{ok}/{len(picked)} sanity prompts look right")

In [ ]:
# Export and upload (full run only): LoRA adapter + GGUF q4_k_m and f16 for llama.cpp.
if MAX_STEPS == -1:
    model.push_to_hub(HF_REPO + "-lora", token=HF_TOKEN, private=False)
    tokenizer.push_to_hub(HF_REPO + "-lora", token=HF_TOKEN, private=False)
    model.push_to_hub_gguf(HF_REPO, tokenizer, quantization_method=["q4_k_m", "f16"], token=HF_TOKEN)
    print("uploaded to https://huggingface.co/" + HF_REPO)
else:
    print("sanity run: nothing uploaded. Set MAX_STEPS = -1 for the full run.")